# 01 : Initial EMG Processing

## Objective
This notebook preprocesses the raw two-channel EMG recordings associated with squatting. It rectifies the signals, calculates rolling root mean square (RMS) values, computes the VMO-to-VL RMS ratio, and assigns preliminary activity categories using predefined thresholds.

## Workflow
1. Load the raw EMG dataset from Google Drive.
2. Rectify both EMG channels using absolute values.
3. Calculate rolling RMS values using a 100-sample window.
4. Compute the VMO-to-VL RMS ratio.
5. Assign preliminary activity categories using the specified thresholds.
6. Save the processed data as a CSV file in Google Drive.

## Output
A processed CSV file containing the original data and derived signal features and classifications.

## Interpretation note
The categories generated by the ratio thresholds are rule-based labels for this analysis. They should not be interpreted as clinically validated diagnoses.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# Step 1: Mount Drive
drive.mount('/content/drive')

# Step 2: Load raw EMG data
file_path = "/content/drive/MyDrive/emg_final.csv"
df = pd.read_csv(file_path)

# Step 3: Rectify (absolute value)
df['Rect_1'] = np.abs(df['Channel_1'])
df['Rect_2'] = np.abs(df['Channel_2'])

# Step 4: RMS function (keeps same length)
def rms(signal, window=100):
    return np.sqrt(np.convolve(signal**2, np.ones(window)/window, mode='same'))

df['RMS_1'] = rms(df['Rect_1'])
df['RMS_2'] = rms(df['Rect_2'])

# Step 5: Compute ratio (VMO / VL)
df['ratio'] = df['RMS_1'] / (df['RMS_2'] + 1e-6)

# Step 6: Classification
def classify(r):
    if r < 0.8:
        return "VL dominant (imbalance)"
    elif r > 1.2:
        return "VMO dominant"
    else:
        return "Balanced"

df['status'] = df['ratio'].apply(classify)

# Step 7: Save processed data
output_path = "/content/drive/MyDrive/emg_processed.csv"
df.to_csv(output_path, index=False)

print("Processed file saved as emg_processed.csv")

Mounted at /content/drive
 Processed file saved as emg_processed.csv
